# 04 — Baseline comparison and synthetic-noise check
Compares SVM, Random Forest and KNN against the CNN using the same held-out speakers. Also runs a basic synthetic-noise test.

In [4]:
from pathlib import Path
import numpy as np, pandas as pd, librosa, tensorflow as tf
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import joblib
OUT=Path("anu_revision_output")
d=np.load(OUT/"speaker_split_mfcc.npz",allow_pickle=True)
Xtr,Xte,ytr,yte=d["X_train"],d["X_test"],d["y_train"],d["y_test"]

In [7]:
a,b=Xtr.reshape(len(Xtr),-1),Xte.reshape(len(Xte),-1)
models={"SVM":SVC(kernel="rbf",class_weight="balanced"),
 "Random Forest":RandomForestClassifier(n_estimators=300,random_state=42,class_weight="balanced"),
 "KNN":KNeighborsClassifier(n_neighbors=5)}
rows=[]
for name,clf in models.items():
 clf.fit(a,ytr)
 p=clf.predict(b)
 pr,re,f1,_=precision_recall_fscore_support(yte,p,average="macro",zero_division=0)
 rows.append({"Model":name,"Accuracy":accuracy_score(yte,p),"Macro precision":pr,"Macro recall":re,"Macro F1":f1})
 joblib.dump(clf,OUT/(name.lower().replace(" ","_")+".joblib"))
cnn=tf.keras.models.load_model(OUT/"anu_cnn_batchnorm_best.keras")
pc=cnn.predict(Xte,verbose=0).argmax(axis=1)
pr,re,f1,_=precision_recall_fscore_support(yte,pc,average="macro",zero_division=0)
rows.append({"Model":"CNN","Accuracy":accuracy_score(yte,pc),"Macro precision":pr,"Macro recall":re,"Macro F1":f1})
result=pd.DataFrame(rows).sort_values("Accuracy",ascending=False)
display(result); result.to_csv(OUT/"baseline_comparison.csv",index=False)

,Model,Accuracy,Macro precision,Macro recall,Macro F1
3,CNN,0.576253,0.591726,0.576623,0.566238
1,Random Forest,0.386710,0.405551,0.386905,0.377821
0,SVM,0.287582,0.310406,0.287879,0.286879
2,KNN,0.285403,0.308426,0.285606,0.291103


In [6]:
SR,N_MFCC,FRAMES=22050,20,40
mean,std=float(d["mean"]),float(d["std"])
def noisy_mfcc(path):
 y,_=librosa.load(path,sr=SR,mono=True)
 y=y+np.random.default_rng(42).normal(0,0.005,len(y))
 x=librosa.feature.mfcc(y=y,sr=SR,n_mfcc=N_MFCC)
 if x.shape[1]<FRAMES: x=np.pad(x,((0,0),(0,FRAMES-x.shape[1])))
 else: x=x[:,:FRAMES]
 return ((x.astype("float32")-mean)/std)[...,None]
Xnoise=np.stack([noisy_mfcc(p) for p in d["test_paths"]])
pn=cnn.predict(Xnoise,verbose=0).argmax(axis=1)
noise_result=pd.DataFrame([{"Condition":"Clean","Accuracy":accuracy_score(yte,pc)},
 {"Condition":"Synthetic noise (std=0.005)","Accuracy":accuracy_score(yte,pn)}])
display(noise_result); noise_result.to_csv(OUT/"noise_test_results.csv",index=False)

,Condition,Accuracy
0,Clean,0.562092
1,Synthetic noise (std=0.005),0.423747


The synthetic-noise check is only a controlled preliminary test, not a substitute for real noisy-room recordings.